# Data Prep
The purpose of this notebook is to transform the data in preparation for the algos. This involves scaling and transforming the numerical data. In addition there will be feature engineering.

In [6]:
import pandas as pd
import numpy as np
from typing import Callable

In [16]:
df = pd.read_parquet('../data/clean-merged-data.parquet')

In [17]:
df.head(1)

,date,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year
0,1939-07-01,76.0,63.0,0.0,19.1,19.4,7,1,1939


In [18]:
def f_to_c(temp_f: float)->float:
    """
    Convert the temperature from farenheit to celcius
    """
    temp_c = (temp_f - 32) * (5/9)

    return temp_c

In [19]:
def convert_to_cyclical(trig_func: Callable, x: float, max_x: float):
    """
    this function is to convert cyclical data to radial data
    """
    radian_value = trig_func(2*np.pi* (x/max_x))

    return radian_value

In [20]:
# convert all temps to celcius
df['tmax_c'] = f_to_c(df.TMAX)
df['tmin_c'] = f_to_c(df.TMIN)

# add the sin and cos cyclical columns for day and month
df['sin_month'] = convert_to_cyclical(np.sin, df.month, 12)
df['cos_month'] = convert_to_cyclical(np.cos, df.month, 12)

# for day we are just assuming every month to 31 
df['sin_day'] = convert_to_cyclical(np.sin, df.day, 31)
df['cos_day'] = convert_to_cyclical(np.cos, df.day, 31)


In [21]:
df.head(1)

,date,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year,tmax_c,tmin_c,sin_month,cos_month,sin_day,cos_day
0,1939-07-01,76.0,63.0,0.0,19.1,19.4,7,1,1939,24.444444,17.222222,-0.5,-0.866025,0.201299,0.97953


In [27]:
df['past_10_day_ave_temp'] = df.SURF_TEMP_C.rolling(window=10,min_periods=1).mean()
df['past_10_day_std_temp'] = df.SURF_TEMP_C.rolling(window=10,min_periods=1).std()

In [28]:
df.head(4)

,date,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year,tmax_c,tmin_c,sin_month,cos_month,sin_day,cos_day,past_10_day_ave_temp,target,past_10_day_std_temp
0,1939-07-01,76.0,63.0,0.0,19.1,19.4,7,1,1939,24.444444,17.222222,-0.5,-0.866025,0.201299,0.979530,19.400000,NaN,NaN
1,1939-07-02,74.0,65.0,0.0,19.5,19.8,7,2,1939,23.333333,18.333333,-0.5,-0.866025,0.394356,0.918958,19.600000,19.4,0.282843
2,1939-07-03,71.0,62.0,0.0,19.5,20.1,7,3,1939,21.666667,16.666667,-0.5,-0.866025,0.571268,0.820763,19.766667,19.8,0.351188
3,1939-07-04,71.0,63.0,0.0,19.3,20.2,7,4,1939,21.666667,17.222222,-0.5,-0.866025,0.724793,0.688967,19.875000,20.1,0.359398


In [29]:
df['target'] = df.SURF_TEMP_C.shift(1)

In [30]:
df.head(5)

,date,TMAX,TMIN,PRCP,BOT_TEMP_C,SURF_TEMP_C,month,day,year,tmax_c,tmin_c,sin_month,cos_month,sin_day,cos_day,past_10_day_ave_temp,target,past_10_day_std_temp
0,1939-07-01,76.0,63.0,0.0,19.1,19.4,7,1,1939,24.444444,17.222222,-0.5,-0.866025,0.201299,0.979530,19.400000,NaN,NaN
1,1939-07-02,74.0,65.0,0.0,19.5,19.8,7,2,1939,23.333333,18.333333,-0.5,-0.866025,0.394356,0.918958,19.600000,19.4,0.282843
2,1939-07-03,71.0,62.0,0.0,19.5,20.1,7,3,1939,21.666667,16.666667,-0.5,-0.866025,0.571268,0.820763,19.766667,19.8,0.351188
3,1939-07-04,71.0,63.0,0.0,19.3,20.2,7,4,1939,21.666667,17.222222,-0.5,-0.866025,0.724793,0.688967,19.875000,20.1,0.359398
4,1939-07-05,72.0,64.0,0.0,17.6,20.6,7,5,1939,22.222222,17.777778,-0.5,-0.866025,0.848644,0.528964,20.020000,20.2,0.449444
